In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import gc

# -----------------------------------------
# 1. Sharpe Ratio Function
# -----------------------------------------
def calc_spread_return_sharpe(df: pd.DataFrame, portfolio_size: int = 200, toprank_weight_ratio: float = 2) -> float:
    def _calc_spread_return_per_day(df, portfolio_size, toprank_weight_ratio):
        assert df['Rank'].min() == 0
        assert df['Rank'].max() == len(df['Rank']) - 1
        n_stocks = len(df)
        if n_stocks < portfolio_size:
            portfolio_size = n_stocks
        top_stocks = df.sort_values(by='Rank')['Target'][:portfolio_size]
        bottom_stocks = df.sort_values(by='Rank', ascending=False)['Target'][:portfolio_size]
        weights = np.linspace(start=toprank_weight_ratio, stop=1, num=len(top_stocks))
        purchase = (top_stocks * weights).sum() / weights.mean()
        short = (bottom_stocks * weights).sum() / weights.mean()
        return purchase - short

    buf = df.groupby('Date').apply(_calc_spread_return_per_day, portfolio_size, toprank_weight_ratio)
    return buf.mean() / (buf.std() + 1e-8)

# -----------------------------------------
# 2. Load Training and Test Data
# -----------------------------------------
feature_file = "training_price_features.csv"
test_file = "validation_price_features.csv"
train_data = pd.read_csv(feature_file)
test_data = pd.read_csv(test_file)

# Preprocess
train_data = train_data.dropna().sort_values(['Date', 'SecuritiesCode']).reset_index(drop=True)
test_data = test_data.dropna().sort_values(['Date', 'SecuritiesCode']).reset_index(drop=True)

# Define features/targets
features = [col for col in train_data.columns if col not in ['Target', 'RowId']]
X_train = train_data[features].drop(columns=['Date'])
y_train = train_data['Target']

X_test = test_data[features].drop(columns=['Date'])
y_test = test_data['Target']

# -----------------------------------------
# 3. Define Best Hyperparameters (from Optuna tuning)
# -----------------------------------------


params = {
        'n_estimators': 500,
        'num_leaves': 100,
        'learning_rate': 0.1,
        'colsample_bytree': 0.9,
        'subsample': 0.8,
        'reg_alpha': 0.4,
        'metric': 'mae',
        'random_state': 21,
        'verbosity': -1  
    }

# -----------------------------------------
# 4. Train Model on Full Training Set
# -----------------------------------------
model = LGBMRegressor(**best_params_v2)
model.fit(X_train, y_train)

# -----------------------------------------
# 5. Predict on Test Set
# -----------------------------------------
y_pred_test = model.predict(X_test)

# Compute Evaluation Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae = mean_absolute_error(y_test, y_pred_test)
print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")

# -----------------------------------------
# 6. Compute Sharpe Ratio on Test Set
# -----------------------------------------
test_df = test_data.copy()
test_df['pred'] = y_pred_test
test_df['Rank'] = test_df.groupby('Date')['pred'].rank(method='first', ascending=False) - 1

test_sharpe = calc_spread_return_sharpe(test_df)
print(f"Test Sharpe Ratio: {test_sharpe:.4f}")

# -----------------------------------------
# 7. Save Submission
# -----------------------------------------
submission = test_df[['Date', 'SecuritiesCode', 'Rank']]
submission.to_csv("submission.csv", index=False)
print("\nSubmission saved to submission.csv")
